# Create Aragón Research Group Awards (Gobierno de Aragón)

Creates Gobierno de Aragón awards from the concession resolutions for the funding of
**recognised research groups** ("grupos de investigación reconocidos por el Gobierno de
Aragón"), published in the Boletín Oficial de Aragón (BOA). One award = one research group
x one three-year funding period. **727 awards: 2017-2019 (235 groups, EUR 9.0M),
2020-2022 (234, EUR 6.0M), 2023-2025 (258, EUR 10.0M)**; 100% title (group name),
PI, host institution, amount (local run 2026-09-30).

Sources (BOA PDFs, Anexo I "Concedidos" only):
- 2017-2019: Resolución de 19 de junio de 2018 (BOA 27/06/2018)
- 2020-2022: Resolución de 17 de agosto de 2021 (BOA 27/08/2021)
- 2023-2025: Resolución de 20 de abril de 2023 (BOA 28/04/2023; per-period total checked
  against the published EUR 9,999,999.91)

**Research filter / scope:** the opendata.aragon.es "convocatorias-ayudas-subvenciones"
dataset lists BOA *call* announcements (no beneficiaries), so it cannot give grant-level
rows. This ingest covers only the research-group programme (I+D+i research groups,
Dirección General de Investigación e Innovación); no non-research subsidies. Other Aragón
I+D calls (predoctoral contracts, R&D projects with companies) are not covered. Annual
2013-2016 group resolutions are skipped: they use bare codes without a period suffix.

**`funder_award_id` (section 2.1.1):** the group reference code with period suffix, exactly
as printed (`T08_20R`, `B05_17D`, `S38_23R`). This is the form grantees cite: it is the
dominant shape among the ~1,450 acknowledgement stubs for F4320326208 (`E46_20R`,
`B08_17R`, `T39 23R`). Only one identifier exists in the source.

**Amount:** subsidy granted to the group for the whole period (EUR). **Dates:** period
start/end (1 Jan of first year, 31 Dec of last year). **PI:** Investigador/a Principal;
co-PI (2023-2025 only) goes to co_lead_investigator. Names are split Spanish-style (two
surnames) in the script.

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320326208`; display_name,
ror_id, doi read from `openalex.funders.funders`.

**Priority:** `522` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

**Prerequisite:** run `scripts/local/aragon_research_groups_to_s3.py` (uploads
`s3://openalex-ingest/awards/aragon_research_groups/aragon_research_groups_projects.parquet`).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aragon_research_groups_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aragon_research_groups/aragon_research_groups_projects.parquet`;

In [ ]:
%sql
SELECT period, COUNT(*) as n, ROUND(SUM(CAST(amount AS DOUBLE)), 2) as total_eur
FROM openalex.awards.aragon_research_groups_raw GROUP BY period ORDER BY period;

In [ ]:
%sql
SELECT * FROM openalex.awards.aragon_research_groups_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320326208 must resolve to exactly 1 row in `openalex.funders.funders`.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320326208 missing from openalex.funders.funders') AS ok
FROM openalex.funders.funders WHERE funder_id = 4320326208;

## Step 2: Create Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aragon_research_groups_awards
USING delta
AS
WITH src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320326208  -- Gobierno de Aragón
),
g AS (
    SELECT *,
        TRIM(funder_award_id) AS award_key,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_dt,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_dt
    FROM openalex.awards.aragon_research_groups_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000 as id,
    NULLIF(TRIM(g.group_name), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.award_key as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('Grupos de investigación reconocidos por el Gobierno de Aragón ', g.period) as funder_scheme,
    'aragon_boa_research_groups' as provenance,
    g.start_dt as start_date,
    g.end_dt as end_date,
    YEAR(g.start_dt) as start_year,
    YEAR(g.end_dt) as end_year,
    CASE WHEN NULLIF(TRIM(g.pi_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.pi_given_name), ''),
        'family_name', NULLIF(TRIM(g.pi_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', g.start_dt,
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'ES',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CASE WHEN NULLIF(TRIM(g.co_pi_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.co_pi_given_name), ''),
        'family_name', NULLIF(TRIM(g.co_pi_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', g.start_dt,
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'ES',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): every id is a period-suffixed group code, unique.
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ids,
       assert_true(SUM(CASE WHEN NOT funder_award_id RLIKE '^[A-Z]{1,3}[0-9]{2,3}_[0-9]{2}[A-Z]$' THEN 1 ELSE 0 END) = 0, 'award id shape') AS id_shape,
       assert_true(SUM(CASE WHEN funder_award_id RLIKE '^360G-' THEN 1 ELSE 0 END) = 0, 'registry id shipped') AS no_registry_ids
FROM openalex.awards.aragon_research_groups_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aragon_boa_research_groups' AND priority = 522;

-- Priority 522: direct-from-funder ingest of Aragón's own BOA concession resolutions.
-- Higher wins under the 2026-06-20 DESC dedup, so 522 outranks the acknowledgement
-- shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    522 as priority
FROM openalex.awards.aragon_research_groups_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.aragon_research_groups_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name,
       co_lead_investigator.family_name AS co_pi_family
FROM openalex.awards.aragon_research_groups_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.aragon_research_groups_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail (a PI can lead the same group in all 3 periods).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.aragon_research_groups_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.aragon_research_groups_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(co_lead_investigator.family_name) as has_co_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.aragon_research_groups_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement stubs for F4320326208 (these upgrade in CreateAwards dedup).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as collapsed_onto_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.aragon_research_groups_awards t ON t.id = c.id
WHERE c.funder_id = 4320326208 AND c.provenance != 'aragon_boa_research_groups'
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aragon_boa_research_groups'
GROUP BY provenance, priority;